# LLM Zoomcamp Homework 1: Agentic RAG

This notebook sets up the environment for the homework using Qwen via an OpenAI-compatible client.

In [ ]:
import json
import os
from pprint import pprint
from dotenv import load_dotenv
from gitsource import GithubRepositoryDataReader, chunk_documents
import minsearch
from openai import OpenAI

In [ ]:
load_dotenv()

QWEN_BASE_URL = os.getenv("QWEN_BASE_URL", "https://dashscope-intl.aliyuncs.com/compatible-mode/v1")
QWEN_MODEL = os.getenv("QWEN_MODEL", "qwen-plus")

client = OpenAI(
    api_key=os.getenv("QWEN_API_KEY"),
    base_url=QWEN_BASE_URL,
)

assert os.getenv("QWEN_API_KEY"), "QWEN_API_KEY is not set"
QWEN_MODEL

## Fetch Course Lessons

Pull the lesson markdown files directly from the course repository at the fixed homework commit.

In [ ]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()
documents = [file.parse() for file in files]

len(documents)

In [ ]:
sample_filenames = [doc["filename"] for doc in documents[:10]]
pprint(sample_filenames)

documents[0].keys()

## Q1: How Many Lesson Pages?

In [ ]:
lesson_page_count = len(documents)
lesson_page_count

## Shared Helpers

In [ ]:
Q2_QUERY = "How does the agentic loop keep calling the model until it stops?"
Q6_QUERY = "How does the agentic loop work, and how is it different from plain RAG?"

INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with \"I don't know.\"
""".strip()

PROMPT_TEMPLATE = """
QUESTION: {question}

CONTEXT:
{context}
""".strip()


def build_index(docs):
    index = minsearch.Index(text_fields=["content"], keyword_fields=["filename"])
    index.fit(docs)
    return index


def build_context(search_results):
    lines = []
    for doc in search_results:
        lines.append(doc["filename"])
        lines.append(doc["content"])
        lines.append("")
    return "\n".join(lines).strip()


def rag_with_usage(index, query, num_results=5):
    search_results = index.search(query, num_results=num_results)
    prompt = PROMPT_TEMPLATE.format(
        question=query,
        context=build_context(search_results),
    )
    response = client.chat.completions.create(
        model=QWEN_MODEL,
        messages=[
            {"role": "system", "content": INSTRUCTIONS},
            {"role": "user", "content": prompt},
        ],
    )
    return {
        "search_results": search_results,
        "prompt": prompt,
        "answer": response.choices[0].message.content,
        "prompt_tokens": response.usage.prompt_tokens,
        "total_tokens": response.usage.total_tokens,
    }


documents_index = build_index(documents)

## Q2: Indexing And Searching

In [ ]:
q2_results = documents_index.search(Q2_QUERY, num_results=5)
q2_results[0]["filename"]

## Q3: Plain RAG

In [ ]:
q3_result = rag_with_usage(documents_index, Q2_QUERY)
q3_result["prompt_tokens"]

In [ ]:
q3_result["answer"]

## Q4: Chunking

In [ ]:
chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)

In [ ]:
chunks[0].keys()

## Q5: RAG With Chunking

In [ ]:
chunks_index = build_index(chunks)
q5_result = rag_with_usage(chunks_index, Q2_QUERY)
q5_result["prompt_tokens"]

In [ ]:
q3_result["prompt_tokens"] / q5_result["prompt_tokens"]

## Q6: Turning It Into An Agent

In [ ]:
AGENT_INSTRUCTIONS = (
    "You're a course teaching assistant. Answer the student's question using the search tool. "
    "Make multiple searches with different keywords before answering."
)


def search_lessons(query: str):
    results = chunks_index.search(query, num_results=5)
    return [
        {
            "filename": r["filename"],
            "start": r.get("start"),
            "content": r["content"][:1200],
        }
        for r in results
    ]


TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "search_lessons",
            "description": "Search the LLM Zoomcamp lesson chunks for relevant course content.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Search query for finding relevant lesson chunks.",
                    }
                },
                "required": ["query"],
            },
        },
    }
]

In [ ]:
messages = [
    {"role": "system", "content": AGENT_INSTRUCTIONS},
    {"role": "user", "content": Q6_QUERY},
]

search_calls = 0
q6_answer = None

for _ in range(10):
    response = client.chat.completions.create(
        model=QWEN_MODEL,
        messages=messages,
        tools=TOOLS,
        tool_choice="auto",
    )
    message = response.choices[0].message

    if getattr(message, "tool_calls", None):
        messages.append(
            {
                "role": "assistant",
                "content": message.content or "",
                "tool_calls": [tool_call.model_dump() for tool_call in message.tool_calls],
            }
        )

        for tool_call in message.tool_calls:
            if tool_call.function.name != "search_lessons":
                continue

            search_calls += 1
            args = json.loads(tool_call.function.arguments)
            tool_result = search_lessons(args["query"])
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(tool_result, ensure_ascii=False),
                }
            )
        continue

    q6_answer = message.content
    break

search_calls

In [ ]:
q6_answer